# 04 - Crear hoplite DB de Train y de Test

## Descripcion

Genera una hoplite DB de Train y otra de Test con los archivos correspondientes definidos en la particion del notebook 04. Para esto crea una una copia de la hoplite DB base (db completa con todos los audios), luego elimina de la db train los archivos de la db test, y de la db test los archivos de la db train. De esta manara se crean db separadas e independientes para entrenamiento y para prueba.  

In [ ]:
from pathlib import Path
import gc
import shutil
import sqlite3

import pandas as pd
import numpy as np

from perch_hoplite.db import sqlite_usearch_impl


# =========================================================
# CONFIGURACIÓN
# =========================================================

COMPLETE_DB_PATH = Path(r"/mnt/d/Taboga/Train/perch_embed")

TRAIN_DB_PATH = Path(r"/mnt/d/Taboga/Train/train_db")

TEST_DB_PATH = Path(r"/mnt/d/Taboga/Train/test_db")

SPLIT_CSV = Path(r"/mnt/d/Taboga/data_splits/recording_split.csv")

# Registro de las anotaciones preliminares retiradas del test.
# Este archivo es únicamente de auditoría; no es el ground truth.
TEST_ANNOTATIONS_AUDIT_CSV = Path("/mnt/d/Taboga/data_splits/test_preliminary_annotations_removed.csv")
TRAIN_ANNOTATIONS_AUDIT_CSV = Path("/mnt/d/Taboga/data_splits/train_preliminary_annotations_removed.csv")

# Cambiar a True cuando se modifique la partición train/test.
REBUILD_SUBSET_DATABASES = True


Cargar la particion

In [ ]:
split_df = pd.read_csv(SPLIT_CSV)

required_columns = {
    "recording_id",
    "split",
}

missing_columns = required_columns.difference(
    split_df.columns
)

if missing_columns:
    raise ValueError(
        "Faltan columnas en recording_split.csv: "
        f"{sorted(missing_columns)}"
    )

split_df["recording_id"] = (
    split_df["recording_id"].astype(int)
)

valid_splits = {"train", "test"}

invalid_splits = set(
    split_df["split"].dropna().unique()
).difference(valid_splits)

if invalid_splits:
    raise ValueError(
        "Se encontraron valores no válidos en la columna split: "
        f"{sorted(invalid_splits)}"
    )

if split_df["recording_id"].duplicated().any():
    duplicated = split_df.loc[
        split_df["recording_id"].duplicated(keep=False),
        ["recording_id", "split"],
    ]

    raise ValueError(
        "Hay recording_id repetidos en recording_split.csv:\n"
        f"{duplicated.head(20)}"
    )

train_recording_ids = set(
    split_df.loc[
        split_df["split"].eq("train"),
        "recording_id",
    ]
)

test_recording_ids = set(
    split_df.loc[
        split_df["split"].eq("test"),
        "recording_id",
    ]
)

if train_recording_ids & test_recording_ids:
    raise RuntimeError(
        "Train y test comparten recording_id."
    )

print(
    f"Grabaciones definidas como train: "
    f"{len(train_recording_ids):,}"
)

print(
    f"Grabaciones definidas como test: "
    f"{len(test_recording_ids):,}"
)

In [ ]:
def close_hoplite_object(
    hoplite_object,
    commit=True,
):
    """
    Guarda y cierra explícitamente un objeto Hoplite.

    También libera la referencia al índice USearch para reducir
    la posibilidad de que el archivo permanezca abierto.
    """

    if hoplite_object is None:
        return

    try:
        if commit:
            hoplite_object.commit()
        else:
            try:
                hoplite_object.db.rollback()
            except Exception:
                pass

    finally:
        try:
            hoplite_object.db.close()
        except Exception:
            pass

        try:
            hoplite_object.ui = None
        except Exception:
            pass

        gc.collect()


def close_open_hoplite_objects():
    """
    Cierra objetos Hoplite conocidos que estén abiertos en este kernel.

    Esto no puede cerrar conexiones abiertas en otros kernels o procesos.
    Antes de ejecutar este notebook también se deben cerrar los notebooks
    de embeddings, pattern matching o Agile Modeling que usen la DB base.
    """

    possible_variable_names = (
        "db",
        "complete_db",
        "source_db",
        "train_db",
        "test_db",
    )

    for variable_name in possible_variable_names:
        hoplite_object = globals().get(variable_name)

        if (
            hoplite_object is not None
            and hasattr(hoplite_object, "db")
            and hasattr(hoplite_object, "commit")
        ):
            print(
                f"Cerrando objeto Hoplite abierto: "
                f"{variable_name}"
            )

            close_hoplite_object(
                hoplite_object,
                commit=True,
            )

            globals()[variable_name] = None

    gc.collect()


def prepare_source_database_for_copy(
    source_db_path,
):
    """
    Ejecuta un checkpoint WAL antes de copiar la base completa.

    El checkpoint incorpora los cambios pendientes dentro de
    hoplite.sqlite. Si otra conexión mantiene una transacción activa,
    la función detiene el proceso en lugar de crear una copia
    potencialmente inconsistente.
    """

    source_db_path = Path(source_db_path)

    sqlite_path = (
        source_db_path
        / "hoplite.sqlite"
    )

    if not sqlite_path.exists():
        raise FileNotFoundError(
            f"No existe la base SQLite: "
            f"{sqlite_path}"
        )

    with sqlite3.connect(
        sqlite_path,
        timeout=60,
    ) as connection:

        connection.execute(
            "PRAGMA busy_timeout = 60000"
        )

        checkpoint_result = connection.execute(
            "PRAGMA wal_checkpoint(TRUNCATE)"
        ).fetchone()

    # El primer valor es distinto de cero cuando el checkpoint
    # no pudo completarse por un bloqueo.
    checkpoint_busy = int(
        checkpoint_result[0]
    )

    if checkpoint_busy != 0:
        raise RuntimeError(
            "No fue posible completar el checkpoint WAL. "
            "Cierre cualquier otro notebook o proceso que "
            "tenga abierta la base completa. "
            f"Resultado: {checkpoint_result}"
        )

    wal_path = Path(
        str(sqlite_path) + "-wal"
    )

    wal_size = (
        wal_path.stat().st_size
        if wal_path.exists()
        else 0
    )

    if wal_size != 0:
        raise RuntimeError(
            "El archivo WAL no quedó vacío después del "
            f"checkpoint. Tamaño: {wal_size:,} bytes."
        )

    print(
        "Base completa preparada para copiar:"
    )
    print(
        f"  Checkpoint WAL: {checkpoint_result}"
    )
    print(
        f"  Tamaño WAL: {wal_size:,} bytes"
    )


def export_and_clear_annotations(
    subset_db,
    annotations_audit_csv,
):
    """
    Exporta todas las anotaciones que permanecen en la copia
    y luego vacía la tabla annotations.

    Debe utilizarse solamente para la base de test.
    """

    annotations_audit_csv = Path(
        annotations_audit_csv
    )

    annotations_audit_csv.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    preliminary_annotations = pd.read_sql_query(
        """
        SELECT
            a.id AS annotation_id,
            a.recording_id,
            r.filename,
            GET_OFFSET_START(a.offsets) AS start_s,
            GET_OFFSET_END(a.offsets) AS end_s,
            a.label,
            a.label_type,
            a.provenance
        FROM annotations AS a
        INNER JOIN recordings AS r
            ON a.recording_id = r.id
        ORDER BY
            a.recording_id,
            start_s,
            annotation_id
        """,
        subset_db.db,
    )

    preliminary_annotations.to_csv(
        annotations_audit_csv,
        index=False,
    )

    removed_annotations_count = len(
        preliminary_annotations
    )

    subset_db.db.execute(
        "DELETE FROM annotations"
    )

    subset_db.commit()

    remaining_annotations = int(
        subset_db.db.execute(
            """
            SELECT COUNT(*)
            FROM annotations
            """
        ).fetchone()[0]
    )

    if remaining_annotations != 0:
        raise RuntimeError(
            "No fue posible eliminar todas las "
            "anotaciones de test. "
            f"Quedan {remaining_annotations:,}."
        )

    print(
        "Anotaciones preliminares exportadas: "
        f"{removed_annotations_count:,}"
    )
    print(
        "Archivo de auditoría: "
        f"{annotations_audit_csv}"
    )
    print(
        "Tabla annotations de test: vacía"
    )

    return removed_annotations_count


def create_filtered_hoplite_db(
    source_db_path,
    target_db_path,
    recording_ids_to_remove,
    expected_recording_ids,
    rebuild=True,
    clear_annotations=False,
    annotations_audit_csv=None,
):
    """
    Copia una base Hoplite y elimina grabaciones completas,
    incluyendo sus ventanas, anotaciones y embeddings.

    La función activa explícitamente las claves foráneas de
    SQLite para que ON DELETE CASCADE funcione.

    Cuando clear_annotations=True, exporta y elimina todas las
    anotaciones que permanecen en la copia. Esta opción debe
    utilizarse únicamente para test.
    """

    source_db_path = Path(source_db_path)
    target_db_path = Path(target_db_path)

    recording_ids_to_remove = {
        int(x) for x in recording_ids_to_remove
    }

    expected_recording_ids = {
        int(x) for x in expected_recording_ids
    }

    if not source_db_path.exists():
        raise FileNotFoundError(
            f"No existe la base original: {source_db_path}"
        )

    if clear_annotations and annotations_audit_csv is None:
        raise ValueError(
            "Debe indicar annotations_audit_csv cuando "
            "clear_annotations=True."
        )

    # ------------------------------------------------------
    # 1. Reconstruir desde una copia limpia
    # ------------------------------------------------------

    if target_db_path.exists():
        if rebuild:
            shutil.rmtree(target_db_path)
            print(
                f"Copia anterior eliminada: "
                f"{target_db_path}"
            )
        else:
            raise FileExistsError(
                f"La base ya existe: {target_db_path}. "
                "Use rebuild=True para reconstruirla."
            )

    # Consolidar el WAL antes de copiar.
    prepare_source_database_for_copy(
        source_db_path
    )

    shutil.copytree(
        source_db_path,
        target_db_path,
        ignore=shutil.ignore_patterns(
            "hoplite.sqlite-wal",
            "hoplite.sqlite-shm",
        ),
    )

    print(
        f"Copia creada: {target_db_path}"
    )

    # ------------------------------------------------------
    # 2. Abrir la copia
    # ------------------------------------------------------

    subset_db = None
    creation_succeeded = False

    try:
        subset_db = (
            sqlite_usearch_impl
            .SQLiteUSearchDB
            .create(
                str(target_db_path)
            )
        )

        # --------------------------------------------------
        # 3. Activar explícitamente foreign keys
        # --------------------------------------------------

        subset_db.db.execute(
            "PRAGMA foreign_keys = ON"
        )

        foreign_keys_enabled = (
            subset_db.db.execute(
                "PRAGMA foreign_keys"
            ).fetchone()[0]
        )

        if foreign_keys_enabled != 1:
            raise RuntimeError(
                "No fue posible activar las claves "
                "foráneas de SQLite."
            )

        print(
            "SQLite foreign keys: ON"
        )

        # --------------------------------------------------
        # 4. Verificar la base antes de modificarla
        # --------------------------------------------------

        initial_window_ids = np.asarray(
            subset_db.match_window_ids(),
            dtype=np.int64,
        )

        initial_contains = np.asarray(
            subset_db.ui.contains(
                initial_window_ids
            ),
            dtype=bool,
        )

        if not initial_contains.all():
            raise RuntimeError(
                "La copia inicial ya contiene "
                "ventanas sin embedding."
            )

        # Liberar arreglos que ya no se necesitan.
        del initial_window_ids
        del initial_contains
        gc.collect()

        # --------------------------------------------------
        # 5. Eliminar grabaciones
        # --------------------------------------------------

        current_recording_ids = {
            int(recording.id)
            for recording
            in subset_db.get_all_recordings()
        }

        ids_to_remove = (
            current_recording_ids
            & recording_ids_to_remove
        )

        missing_removal_ids = (
            recording_ids_to_remove
            - current_recording_ids
        )

        if missing_removal_ids:
            raise RuntimeError(
                f"{len(missing_removal_ids)} IDs que "
                "debían eliminarse no existen en la "
                "base completa. Primeros: "
                f"{sorted(missing_removal_ids)[:10]}"
            )

        print(
            f"Eliminando "
            f"{len(ids_to_remove):,} "
            "grabaciones..."
        )

        for i, recording_id in enumerate(
            sorted(ids_to_remove),
            start=1,
        ):
            subset_db.remove_recording(
                recording_id
            )

            if i % 100 == 0:
                print(
                    f"  Procesadas {i:,} de "
                    f"{len(ids_to_remove):,}"
                )

        subset_db.commit()

        # --------------------------------------------------
        # 6. Limpiar anotaciones del test, si corresponde
        # --------------------------------------------------

        removed_annotations_count = 0

        if clear_annotations:
            removed_annotations_count = (
                export_and_clear_annotations(
                    subset_db,
                    annotations_audit_csv,
                )
            )

        # --------------------------------------------------
        # 7. Verificar filas huérfanas
        # --------------------------------------------------

        orphan_windows = subset_db.db.execute(
            """
            SELECT COUNT(*)
            FROM windows AS w
            LEFT JOIN recordings AS r
                ON w.recording_id = r.id
            WHERE r.id IS NULL
            """
        ).fetchone()[0]

        orphan_annotations = subset_db.db.execute(
            """
            SELECT COUNT(*)
            FROM annotations AS a
            LEFT JOIN recordings AS r
                ON a.recording_id = r.id
            WHERE r.id IS NULL
            """
        ).fetchone()[0]

        if orphan_windows != 0:
            raise RuntimeError(
                f"La base contiene "
                f"{orphan_windows:,} "
                "ventanas huérfanas."
            )

        if orphan_annotations != 0:
            raise RuntimeError(
                f"La base contiene "
                f"{orphan_annotations:,} "
                "anotaciones huérfanas."
            )

        # --------------------------------------------------
        # 8. Verificar recordings esperadas
        # --------------------------------------------------

        remaining_recording_ids = {
            int(recording.id)
            for recording
            in subset_db.get_all_recordings()
        }

        missing_expected = (
            expected_recording_ids
            - remaining_recording_ids
        )

        unexpected = (
            remaining_recording_ids
            - expected_recording_ids
        )

        if missing_expected:
            raise RuntimeError(
                f"Faltan "
                f"{len(missing_expected):,} "
                "grabaciones esperadas."
            )

        if unexpected:
            raise RuntimeError(
                f"Quedan "
                f"{len(unexpected):,} "
                "grabaciones inesperadas."
            )

        # --------------------------------------------------
        # 9. Verificar sincronización SQLite-USearch
        # --------------------------------------------------

        remaining_window_ids = np.asarray(
            subset_db.match_window_ids(),
            dtype=np.int64,
        )

        contains_mask = np.asarray(
            subset_db.ui.contains(
                remaining_window_ids
            ),
            dtype=bool,
        )

        n_sqlite = len(
            remaining_window_ids
        )

        n_usearch = (
            subset_db.count_embeddings()
        )

        n_missing = int(
            (~contains_mask).sum()
        )

        n_annotations = int(
            subset_db.db.execute(
                """
                SELECT COUNT(*)
                FROM annotations
                """
            ).fetchone()[0]
        )

        print(
            "\nVerificación final"
        )
        print(
            f"  Grabaciones SQLite: "
            f"{len(remaining_recording_ids):,}"
        )
        print(
            f"  Ventanas SQLite: "
            f"{n_sqlite:,}"
        )
        print(
            f"  Embeddings USearch: "
            f"{n_usearch:,}"
        )
        print(
            f"  Ventanas sin embedding: "
            f"{n_missing:,}"
        )
        print(
            f"  Ventanas huérfanas: "
            f"{orphan_windows:,}"
        )
        print(
            f"  Anotaciones huérfanas: "
            f"{orphan_annotations:,}"
        )
        print(
            f"  Anotaciones presentes: "
            f"{n_annotations:,}"
        )

        if n_missing != 0:
            missing_ids = (
                remaining_window_ids[
                    ~contains_mask
                ]
            )

            raise RuntimeError(
                "SQLite y USearch no están "
                "sincronizados. Primeros IDs "
                "faltantes: "
                f"{missing_ids[:10].tolist()}"
            )

        if n_sqlite != n_usearch:
            raise RuntimeError(
                "El número de ventanas SQLite "
                "no coincide con el número de "
                "embeddings USearch."
            )

        if (
            clear_annotations
            and n_annotations != 0
        ):
            raise RuntimeError(
                "La base de test todavía "
                f"contiene {n_annotations:,} "
                "anotaciones."
            )

        creation_report = {
            "db_path": str(
                target_db_path
            ),
            "n_recordings": len(
                remaining_recording_ids
            ),
            "n_windows": int(
                n_sqlite
            ),
            "n_embeddings": int(
                n_usearch
            ),
            "n_annotations": int(
                n_annotations
            ),
            "removed_annotations": int(
                removed_annotations_count
            ),
        }

        subset_db.commit()
        creation_succeeded = True

        print(
            f"\nBase creada correctamente: "
            f"{target_db_path}"
        )

        return creation_report

    finally:
        if subset_db is not None:
            close_hoplite_object(
                subset_db,
                commit=creation_succeeded,
            )

        subset_db = None
        gc.collect()

        print(
            f"Conexión cerrada: "
            f"{target_db_path}"
        )


In [ ]:
# =========================================================
# CERRAR OBJETOS HOPLITE ABIERTOS EN ESTE KERNEL
# =========================================================

close_open_hoplite_objects()

print("Los objetos Hoplite conocidos de este kernel fueron cerrados.")

print("Recuerde cerrar también otros kernels o procesos que estén utilizando la base completa.")


Crear la base de datos de TRAIN

In [ ]:
train_report = create_filtered_hoplite_db(
    source_db_path=COMPLETE_DB_PATH,
    target_db_path=TRAIN_DB_PATH,
    recording_ids_to_remove=test_recording_ids,
    expected_recording_ids=train_recording_ids,
    rebuild=REBUILD_SUBSET_DATABASES,
    clear_annotations=True,
    annotations_audit_csv = TRAIN_ANNOTATIONS_AUDIT_CSV
)

train_report


Crear la DB de Test

In [ ]:
test_report = create_filtered_hoplite_db(
    source_db_path=COMPLETE_DB_PATH,
    target_db_path=TEST_DB_PATH,
    recording_ids_to_remove=train_recording_ids,
    expected_recording_ids=test_recording_ids,
    rebuild=REBUILD_SUBSET_DATABASES,
    clear_annotations=True,
    annotations_audit_csv= TEST_ANNOTATIONS_AUDIT_CSV
)

test_report


## Resultado esperado

- `train_db` conserva las anotaciones utilizadas para Agile Modeling.
- `test_db` queda con la tabla `annotations` vacía.
- Las anotaciones preliminares retiradas del test se guardan en
  `test_preliminary_annotations_removed.csv`.
- Las dos conexiones se cierran explícitamente al terminar su construcción.

Las anotaciones manuales de Raven deben almacenarse posteriormente en un
archivo independiente; el CSV de auditoría no constituye el *ground truth*.